# A/B + GPT-6 Luna 공통 생성·전체 시간·API 사용량
현재 설정은 **실제 API 파일럿**이다. NQ·HotpotQA 각각 1문항을 A/B로 실행하며 호출 한도는 4회다.
프로젝트 루트 `.env`에 `OPENAI_API_KEY`를 설정한다. `RUN_API=False`로 바꾸면 실제 호출을 건너뛴다.
검색 캐시를 사용하지 않으며 API 호출 한도는 질문 수 × 두 방법 × 반복으로 검증한다.
짧은 정답 라벨과 비교할 수 있도록 공통 프롬프트는 `answer`에 최소 정답 단어·숫자·구만 요구한다.

### 1. 공통 설정

In [ ]:
from pathlib import Path
import sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'config/baselines.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config/baselines.json').exists(), '프로젝트 루트에서 열어주세요.'
from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=False, encoding='utf-8-sig')
sys.path.insert(0, str(ROOT))
from rag_baselines.data import read_json, read_jsonl, file_sha256, write_json, write_jsonl
import pandas as pd
pd.set_option('display.max_colwidth', 90)
CONFIG = read_json(ROOT / 'config/baselines.json')
print('Workspace:', ROOT)

In [ ]:
import os
RUN_API = True
SPLIT = 'dev'
QUESTIONS_PER_TASK = 1
REPEATS = 1
MAX_API_CALLS = 4
RETRIEVE_K, EVIDENCE_K = 50, 5
CORPUS_SIZE = None
generation_config = read_json(ROOT/'config/generation.json')
print('Model:', generation_config['model'])
print('Reasoning:',generation_config['reasoning_effort'],'output limit:',generation_config['max_output_tokens'])
print('API enabled:',RUN_API,'key configured:',bool(os.environ.get('OPENAI_API_KEY')))
display(pd.DataFrame([generation_config['prices']]))

### 2. 실제 시스템 실행
정답은 검색·생성 후 평가에만 쓴다. 요청마다 원문·원시 응답·반환 모델명·usage·캐시 읽기/쓰기·가격 출처를 저장한다.
SDK 자동 재시도는 0회다. 네트워크 오류를 성공으로 덮지 않는다.

In [ ]:
system_results = []
if RUN_API:
    from openai import OpenAI
    from rag_baselines.models import Baselines
    from rag_baselines.evaluation import measure_system, runtime_manifest, save_run
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('프로젝트 루트 .env에 OPENAI_API_KEY를 설정한 뒤 커널을 다시 시작하세요.')
    manifest = read_json(ROOT/'data/prepared/manifest.json')
    size = CORPUS_SIZE or manifest['n0']
    entry = manifest['corpora'][str(size)]
    assert file_sha256(ROOT/entry['path']) == entry['sha256']
    all_questions = read_jsonl(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')
    questions = [q for task in CONFIG['qa']['tasks'] for q in [r for r in all_questions if r['task']==task][:QUESTIONS_PER_TASK]]
    labels = read_jsonl(ROOT/f'data/prepared/labels_{SPLIT}.jsonl')
    engine = Baselines(ROOT,retrieve_k=RETRIEVE_K,evidence_k=EVIDENCE_K)
    engine.build_index(ROOT/entry['path'])
    engine.warmup(questions[0]['question'])
    client = OpenAI(max_retries=0,timeout=180)
    system_results = measure_system(engine,questions,labels,client=client,generation_config=generation_config,max_calls=MAX_API_CALLS,repeats=REPEATS)
    meta = runtime_manifest(ROOT,engine)
    meta.update({'mode':'live_system','generation':generation_config,'split':SPLIT,'repeats':REPEATS})
    run_directory = save_run(ROOT,[r['retrieval'] for r in system_results],meta,'03_measure_system_api.ipynb')
    write_jsonl(run_directory/'system.jsonl',system_results)
    print('Saved:',run_directory)
else:
    print('미실행: 실제 답변·API latency·usage·비용은 아직 측정되지 않았습니다.')

### 3. 질문별·요약 결과

In [ ]:
if system_results:
    rows = []
    for r in system_results:
        g = r['generation']
        usage = g['api_usage'] or {}
        rows.append({'question_id':r['retrieval']['question_id'],'task':r['retrieval']['task'],
                     'method':r['retrieval']['method'],'repeat':r['retrieval']['repeat'],
                     'online_total_ms':r['online_total_seconds']*1000,'answer':g['answer'],
                     'valid_output':g['valid_output'],'input_tokens':usage.get('input_tokens'),
                     'output_tokens':usage.get('output_tokens'),'api_cost_usd':g['api_cost_usd'],**r['answer_metrics']})
    system_table = pd.DataFrame(rows)
    system_table.to_csv(run_directory/'system.csv',index=False)
    display(system_table)
    system_summary = system_table.groupby(['task','method']).agg(
        questions=('question_id','nunique'),answer_em=('answer_em','mean'),answer_f1=('answer_f1','mean'),
        p50_ms=('online_total_ms','median'),cost_usd=('api_cost_usd','sum'))
    system_summary.to_csv(run_directory/'system_summary.csv')
    display(system_summary)
else:
    print('No measured API results.')

가격은 2026-10-03의 Standard/global 기준 usage 추정액이며 청구서가 아니다. 로컬 GPU 비용은 별도다.
API 모델 alias는 서버에서 갱신될 수 있으므로 returned model·실행 날짜를 기록한다.
o200k_base는 로컬 텍스트 예산용 proxy이며 청구 토큰은 API usage만 사용한다.
[모델](https://developers.openai.com/api/docs/models/gpt-6-luna) ·
[가격](https://developers.openai.com/api/docs/pricing) ·
[캐시 usage](https://developers.openai.com/api/docs/guides/prompt-caching)